In [ ]:
# ============================================================
# CELL 1 — DATA LOADING
# Source: Lens.org | Countries: Colombia, Brazil, Vietnam, Malaysia
# Period: 2020-2025 | Total: 265,004 records
# ============================================================

import pandas as pd
import numpy as np
import glob
import warnings
warnings.filterwarnings('ignore')

# Column positions by file type
# Brazil has 37 columns (broken encoding), others have 35-36
POS_MAP_BRAZIL = {
    6:  'pub_year',
    20: 'cited_by_count',
    21: 'simple_family_size',
    23: 'family_jurisdictions',
    24: 'extended_family_size',
    28: 'cpc_classifications',
    36: 'legal_status',
}

POS_MAP_REST = {
    6:  'pub_year',
    19: 'cited_by_count',
    20: 'simple_family_size',
    22: 'family_jurisdictions',
    23: 'extended_family_size',
    27: 'cpc_classifications',
    35: 'legal_status',
}

COLS_NEEDED = [
    'simple_family_size',
    'extended_family_size',
    'family_jurisdictions',
    'cited_by_count',
    'legal_status',
    'cpc_classifications',
    'pub_year'
]

def detect_pos_map(df_raw):
    return POS_MAP_BRAZIL if len(df_raw.columns) >= 37 else POS_MAP_REST

def load_file(filepath):
    df = pd.read_csv(filepath, sep=',', encoding='latin-1',
                     low_memory=False, header=0)
    pos_map = detect_pos_map(df)
    cols = list(df.columns)
    rename_dict = {cols[pos]: name
                   for pos, name in pos_map.items()
                   if pos < len(cols)}
    df = df.rename(columns=rename_dict)
    cols_present = [c for c in COLS_NEEDED if c in df.columns]
    return df[cols_present].copy()

def load_country(name, file_list):
    frames = []
    for filepath in file_list:
        try:
            df = load_file(filepath)
            df['country'] = name
            frames.append(df)
            print(f"   {filepath}: {len(df):,} records")
        except Exception as e:
            print(f"   {filepath}: {e}")
    if not frames:
        print(f" Failed to load {name}")
        return pd.DataFrame()
    result = pd.concat(frames, ignore_index=True)
    print(f"   {name} total: {len(result):,} records\n")
    return result

# File paths
brazil_files = sorted(glob.glob('Brazil_Patentes_Lens_*.csv'))

FILES = {
    'Colombia': ['Colombia_Patentes_Lens.csv'],
    'Brazil':   brazil_files,
    'Vietnam':  ['Vietman_Patentes_Lens.csv'],
    'Malaysia': ['Malasya_Patentes_Lens.csv'],
}

# Load all countries
print("Loading data...\n")
country_frames = []
for country, files in FILES.items():
    df_c = load_country(country, files)
    if not df_c.empty:
        country_frames.append(df_c)

df = pd.concat(country_frames, ignore_index=True)
print(f"Total dataset: {len(df):,} rows")
print(df.groupby('country').size().rename('n_patents'))

# Quick verification — Brazil family size must not be zero
print("\n Brazil family size check:")
print(df[df['country']=='Brazil']['simple_family_size']
      .apply(pd.to_numeric, errors='coerce')
      .describe().round(2))

In [ ]:
# CELL 2 — PREPROCESSING AND DESCRIPTIVE STATISTICS

# Jurisdictional coverage: count jurisdictions
def count_jurisdictions(val):
    if pd.isna(val) or val == '':
        return 0
    return len(str(val).split(';'))

df['jurisdiction_count'] = df['family_jurisdictions'].apply(count_jurisdictions)

# Legal status -> numerical score
LEGAL_MAP = {
    'active': 1.0, 'granted': 1.0, 'alive': 1.0, 'patented': 1.0,
    'pending': 0.5, 'application': 0.5,
    'expired': 0.25, 'lapsed': 0.25, 'discontinued': 0.25,
    'dead': 0.0, 'abandoned': 0.0, 'revoked': 0.0, 'inactive': 0.0,
}

def map_legal(val):
    if pd.isna(val):
        return 0.5
    v = str(val).lower().strip()
    for key, score in LEGAL_MAP.items():
        if key in v:
            return score
    return 0.5

df['legal_score'] = df['legal_status'].apply(map_legal)

# CPC primary section
def extract_cpc_section(val):
    if pd.isna(val) or val == '':
        return 'Unknown'
    return str(val)[0].upper()

df['cpc_section'] = df['cpc_classifications'].apply(extract_cpc_section)

# Clean numerical columns
for col in ['simple_family_size', 'extended_family_size',
            'cited_by_count', 'jurisdiction_count']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

# Descriptive statistics
print("Descriptive statistics by country:\n")
summary = df.groupby('country').agg(
    N=('simple_family_size', 'count'),
    FS_mean=('simple_family_size', 'mean'),
    FS_median=('simple_family_size', 'median'),
    FE_mean=('extended_family_size', 'mean'),
    JC_mean=('jurisdiction_count', 'mean'),
    CA_mean=('cited_by_count', 'mean'),
    CA_median=('cited_by_count', 'median'),
    LS_mean=('legal_score', 'mean'),
).round(3)
print(summary.to_string())

# Legal status distribution
print("\nLegal status distribution by country:")
legal_dist = df.groupby(['country', 'legal_status']).size().unstack(fill_value=0)
print(legal_dist.to_string())

# Top 5 CPC sections by country
print("\nTop 5 CPC sections by country:")
for country in df['country'].unique():
    top = df[df['country']==country]['cpc_section'].value_counts().head(5)
    print(f"\n{country}:")
    print(top.to_string())

In [ ]:
# CELL 3 — MAMDANI FUZZY LOGIC MODEL
# Outputs: Commercial Impact Potential (CIP) and Academic Impact Potential (AIP)

!pip install scikit-fuzzy -q

import skfuzzy as fuzz
from skfuzzy import control as ctrl

def compute_country_scores(df_country, country_name):
    sample = df_country.sample(min(1500, len(df_country)), random_state=42)

    fs = np.log1p(sample['simple_family_size'].values)
    fe = np.log1p(sample['extended_family_size'].values)
    jc = np.log1p(sample['jurisdiction_count'].values)
    ca = np.log1p(sample['cited_by_count'].values)
    ls = sample['legal_score'].values

    def make_universe(x):
        return np.linspace(0, max(x.max(), 0.01), 200)

    def make_trimf_vars(universe, data):
        p10, p50, p90 = np.percentile(data, [10, 50, 90])
        var = ctrl.Antecedent(universe, 'v')
        var['low']  = fuzz.trimf(universe, [universe[0], universe[0], p50])
        var['mid']  = fuzz.trimf(universe, [p10, p50, p90])
        var['high'] = fuzz.trimf(universe, [p50, universe[-1], universe[-1]])
        return var

    u_fs  = make_universe(fs)
    u_fe  = make_universe(fe)
    u_jc  = make_universe(jc)
    u_ca  = make_universe(ca)
    u_ls  = np.linspace(0, 1, 100)
    u_out = np.linspace(0, 10, 100)

    v_fs = make_trimf_vars(u_fs, fs); v_fs.label = 'family_simple'
    v_fe = make_trimf_vars(u_fe, fe); v_fe.label = 'family_extended'
    v_jc = make_trimf_vars(u_jc, jc); v_jc.label = 'jurisdiction'
    v_ca = make_trimf_vars(u_ca, ca); v_ca.label = 'citations'

    v_ls = ctrl.Antecedent(u_ls, 'legal')
    v_ls['low']  = fuzz.trimf(u_ls, [0, 0, 0.4])
    v_ls['mid']  = fuzz.trimf(u_ls, [0.25, 0.5, 0.75])
    v_ls['high'] = fuzz.trimf(u_ls, [0.6, 1, 1])

    cip = ctrl.Consequent(u_out, 'commercial_impact')
    iap = ctrl.Consequent(u_out, 'academic_impact')
    for out in [cip, iap]:
        out['low']  = fuzz.trimf(u_out, [0, 0, 4])
        out['mid']  = fuzz.trimf(u_out, [2, 5, 8])
        out['high'] = fuzz.trimf(u_out, [6, 10, 10])

    # CIP rules
    rules_cip = [
        ctrl.Rule(v_jc['high'] & v_fe['high'] & v_ls['high'], cip['high']),
        ctrl.Rule(v_jc['high'] & v_fe['mid']  & v_ls['high'], cip['high']),
        ctrl.Rule(v_jc['mid']  & v_fe['mid']  & v_ls['high'], cip['mid']),
        ctrl.Rule(v_jc['mid']  & v_fe['low']  & v_ls['mid'],  cip['mid']),
        ctrl.Rule(v_jc['low']  & v_fe['low']  & v_ls['low'],  cip['low']),
        ctrl.Rule(v_fs['high'] & v_jc['high'],                cip['high']),
        ctrl.Rule(v_fs['low']  & v_jc['low'],                 cip['low']),
    ]

    # AIP rules
    rules_aip = [
        ctrl.Rule(v_ca['high'] & v_fs['high'], iap['high']),
        ctrl.Rule(v_ca['high'] & v_fs['mid'],  iap['high']),
        ctrl.Rule(v_ca['mid']  & v_fs['mid'],  iap['mid']),
        ctrl.Rule(v_ca['mid']  & v_fs['low'],  iap['mid']),
        ctrl.Rule(v_ca['low']  & v_fs['low'],  iap['low']),
        ctrl.Rule(v_ca['high'] & v_jc['low'],  iap['high']),
        ctrl.Rule(v_ca['low']  & v_ls['low'],  iap['low']),
    ]

    sys_cip = ctrl.ControlSystem(rules_cip)
    sys_aip = ctrl.ControlSystem(rules_aip)

    scores_cip, scores_aip = [], []

    for _, row in sample.iterrows():
        try:
            sim = ctrl.ControlSystemSimulation(sys_cip)
            sim.input['family_simple']   = np.clip(np.log1p(row['simple_family_size']),   u_fs[0], u_fs[-1])
            sim.input['family_extended'] = np.clip(np.log1p(row['extended_family_size']), u_fe[0], u_fe[-1])
            sim.input['jurisdiction']    = np.clip(np.log1p(row['jurisdiction_count']),   u_jc[0], u_jc[-1])
            sim.input['legal']           = np.clip(row['legal_score'], 0, 1)
            sim.compute()
            scores_cip.append(sim.output['commercial_impact'])
        except:
            pass

        try:
            sim2 = ctrl.ControlSystemSimulation(sys_aip)
            sim2.input['family_simple'] = np.clip(np.log1p(row['simple_family_size']),  u_fs[0], u_fs[-1])
            sim2.input['citations']     = np.clip(np.log1p(row['cited_by_count']),      u_ca[0], u_ca[-1])
            sim2.input['jurisdiction']  = np.clip(np.log1p(row['jurisdiction_count']),  u_jc[0], u_jc[-1])
            sim2.input['legal']         = np.clip(row['legal_score'], 0, 1)
            sim2.compute()
            scores_aip.append(sim2.output['academic_impact'])
        except:
            pass

    cip_score = round(np.mean(scores_cip), 3) if scores_cip else 0
    aip_score = round(np.mean(scores_aip), 3) if scores_aip else 0
    print(f"  {country_name:12s} -> CIP: {cip_score:.3f} | AIP: {aip_score:.3f}")
    return cip_score, aip_score

print("Computing fuzzy scores (2-3 minutes)...\n")
results = {}
for country in ['Colombia', 'Brazil', 'Vietnam', 'Malaysia']:
    df_c = df[df['country'] == country].copy()
    cip, aip = compute_country_scores(df_c, country)
    results[country] = {'CIP': cip, 'AIP': aip}

df_results = pd.DataFrame(results).T
print("\nFinal scores:")
print(df_results.to_string())

In [ ]:
# CELL 4 — CONVERGENT VALIDATION WITH WORLD BANK INDICATORS

import requests

COUNTRIES_WB = {
    'CO': 'Colombia',
    'BR': 'Brazil',
    'VN': 'Vietnam',
    'MY': 'Malaysia'
}

INDICATORS = {
    'NY.GDP.PCAP.CD':    'GDP per capita (USD)',
    'IP.PAT.RESD':       'Patent applications (residents)',
    'GB.XPD.RSDV.GD.ZS': 'R&D expenditure (% GDP)',
    'NY.GDP.MKTP.KD.ZG': 'GDP growth (%)',
}

def fetch_indicator(iso, indicator, start=2020, end=2024):
    url = (f'https://api.worldbank.org/v2/country/{iso}'
           f'/indicator/{indicator}'
           f'?date={start}:{end}&format=json&per_page=100')
    try:
        data = requests.get(url, timeout=15).json()
        if len(data) < 2 or not data[1]:
            return None
        values = [d['value'] for d in data[1] if d['value'] is not None]
        return round(sum(values)/len(values), 3) if values else None
    except:
        return None

print("Fetching World Bank data...\n")
rows = []
for iso, name in COUNTRIES_WB.items():
    row = {'country': name}
    for code, label in INDICATORS.items():
        val = fetch_indicator(iso, code)
        row[label] = val
        status = f"{val:,.2f}" if val else "no data"
        print(f"  {name:10s} | {label:35s} | {status}")
    rows.append(row)
    print()

df_macro = pd.DataFrame(rows).set_index('country')

# Merge with fuzzy scores
df_val = df_macro.join(df_results)
print("Combined table (macro + fuzzy scores):")
print(df_val.to_string())

# Pearson correlations
print("\nPearson correlations with fuzzy scores:")
macro_cols = ['GDP per capita (USD)', 'R&D expenditure (% GDP)', 'GDP growth (%)']
for col in macro_cols:
    temp = df_val[['CIP', 'AIP', col]].dropna()
    if len(temp) >= 3:
        r_cip = temp['CIP'].corr(temp[col])
        r_aip = temp['AIP'].corr(temp[col])
        print(f"  {col:35s} | r(CIP)={r_cip:.3f} | r(AIP)={r_aip:.3f} | n={len(temp)}")
    else:
        print(f"  {col:35s} | insufficient data (n={len(temp)})")

In [ ]:
# CELL 5 — IEEE-COMPLIANT FIGURES (B&W, 300 DPI, Times New Roman 8pt)

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.rcParams.update({
    'font.family':      'serif',
    'font.size':         8,
    'axes.titlesize':    8,
    'axes.labelsize':    8,
    'xtick.labelsize':   7,
    'ytick.labelsize':   7,
    'legend.fontsize':   7,
    'figure.dpi':        300,
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
    'axes.linewidth':    0.5,
    'lines.linewidth':   0.8,
    'patch.linewidth':   0.5,
})

COUNTRIES = ['Colombia', 'Brazil', 'Vietnam', 'Malaysia']
CIP_VALS  = [5.384, 5.023, 4.976, 5.013]
AIP_VALS  = [4.517, 4.518, 5.378, 4.518]
HATCHES   = ['', '///', 'xxx', '...']
FILLS     = ['black', 'white', 'dimgray', 'lightgray']
MARKERS   = ['o', 's', '^', 'D']

# --- Figure 1: CIP vs AIP bar chart ---
fig, ax = plt.subplots(figsize=(3.5, 3.0))
x = np.arange(len(COUNTRIES))
w = 0.35

for i, (c, cip, aip, hatch, fill) in enumerate(
        zip(COUNTRIES, CIP_VALS, AIP_VALS, HATCHES, FILLS)):
    ax.bar(x[i]-w/2, cip, w, color=fill, edgecolor='black',
           linewidth=0.5, hatch=hatch)
    ax.bar(x[i]+w/2, aip, w, color='white', edgecolor='black',
           linewidth=0.5, hatch=hatch+'\\\\')
    ax.text(x[i]-w/2, cip+0.05, f'{cip:.2f}', ha='center',
            va='bottom', fontsize=6)
    ax.text(x[i]+w/2, aip+0.05, f'{aip:.2f}', ha='center',
            va='bottom', fontsize=6)

ax.axhline(5.0, color='black', linewidth=0.6, linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels(COUNTRIES, fontsize=7)
ax.set_ylabel('Fuzzy Score (0-10)')
ax.set_ylim(4.0, 6.2)
ax.set_title('(a) CIP and AIP Scores by Country')
patch_cip = mpatches.Patch(facecolor='gray', edgecolor='black',
                            linewidth=0.5, label='CIP')
patch_aip = mpatches.Patch(facecolor='white', edgecolor='black',
                            linewidth=0.5, hatch='\\\\\\\\', label='AIP')
line_mid  = plt.Line2D([0],[0], color='black', linewidth=0.6,
                        linestyle='--', label='Midpoint (5.0)')
ax.legend(handles=[patch_cip, patch_aip, line_mid],
          loc='upper right', frameon=True,
          edgecolor='black', fancybox=False)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig1_scores.png', dpi=300)
plt.savefig('fig1_scores.eps')
plt.show()
print("Figure 1 saved")

# --- Figure 2: Scatter CIP vs AIP ---
fig, ax = plt.subplots(figsize=(3.5, 3.2))

JITTER = {
    'Colombia': (0.0,   0.0),
    'Brazil':   (-0.04, -0.02),
    'Vietnam':  (0.0,   0.0),
    'Malaysia': (0.04,  0.02),
}
LABEL_OFFSET = {
    'Colombia': (0.04,  0.02),
    'Brazil':   (-0.18, -0.06),
    'Vietnam':  (0.04,  0.03),
    'Malaysia': (0.04,  0.02),
}

for c, cip, aip, marker in zip(COUNTRIES, CIP_VALS, AIP_VALS, MARKERS):
    jx, jy = JITTER[c]
    lx, ly = LABEL_OFFSET[c]
    ax.scatter(cip+jx, aip+jy, marker=marker, s=55,
               color='black', zorder=5)
    ax.annotate(c, xy=(cip+jx, aip+jy),
                xytext=(cip+jx+lx, aip+jy+ly), fontsize=6.5)

ax.axhline(5.0, color='black', linewidth=0.5, linestyle='--')
ax.axvline(5.0, color='black', linewidth=0.5, linestyle='--')
ax.text(5.70, 5.70, 'High CIP\nHigh AIP', fontsize=5.5,
        ha='center', style='italic', color='dimgray')
ax.text(4.72, 5.70, 'Low CIP\nHigh AIP', fontsize=5.5,
        ha='center', style='italic', color='dimgray')
ax.text(5.70, 4.35, 'High CIP\nLow AIP', fontsize=5.5,
        ha='center', style='italic', color='dimgray')
ax.text(4.72, 4.35, 'Low CIP\nLow AIP', fontsize=5.5,
        ha='center', style='italic', color='dimgray')
ax.text(4.52, 4.28,
        '*Brazil and Malaysia scores nearly overlap;\n'
        ' small offset applied for visibility.',
        fontsize=5, style='italic', color='gray')

for c, marker in zip(COUNTRIES, MARKERS):
    ax.scatter([], [], marker=marker, color='black', s=35, label=c)
ax.legend(loc='lower right', frameon=True,
          edgecolor='black', fancybox=False, fontsize=6)
ax.set_xlabel('Commercial Impact Potential (CIP)')
ax.set_ylabel('Academic Impact Potential (AIP)')
ax.set_title('(b) Portfolio Positioning: CIP vs. AIP')
ax.set_xlim(4.45, 6.05)
ax.set_ylim(4.20, 6.05)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig2_scatter.png', dpi=300)
plt.savefig('fig2_scatter.eps')
plt.show()
print("Figure 2 saved")

# --- Figure 3: CPC Distribution ---
CPC_DATA = {
    'Colombia': {'A':26.7,'C':26.2,'H':7.1,'B':7.0,'G':5.0,'F':4.0,'Other':24.0},
    'Brazil':   {'A':25.8,'C':18.9,'H':10.6,'B':15.5,'G':8.7,'F':5.0,'Other':15.5},
    'Vietnam':  {'A':17.4,'C':22.6,'H':11.9,'B':3.6,'G':15.6,'F':4.0,'Other':24.9},
    'Malaysia': {'A':40.0,'C':21.4,'H':11.0,'B':13.1,'G':5.0,'F':3.0,'Other':6.5},
}
SECTIONS = ['A','B','C','G','H','F','Other']
BW_FILLS  = ['black','dimgray','gray','darkgray','silver','lightgray','white']
BW_HATCH  = ['','//','xx','..','--','||','++']

fig, ax = plt.subplots(figsize=(3.5, 3.6))
x       = np.arange(len(COUNTRIES))
bottoms = np.zeros(len(COUNTRIES))

for sec, fill, hatch in zip(SECTIONS, BW_FILLS, BW_HATCH):
    vals = [CPC_DATA[c].get(sec, 0) for c in COUNTRIES]
    ax.bar(x, vals, bottom=bottoms, color=fill, edgecolor='black',
           linewidth=0.4, hatch=hatch, label=sec)
    bottoms += np.array(vals)

ax.set_xticks(x)
ax.set_xticklabels(COUNTRIES, fontsize=7)
ax.set_ylabel('Share (%)')
ax.set_ylim(0, 105)
ax.set_title('(c) Technological Specialization by CPC Section')
ax.legend(title='CPC Section',
          loc='upper center',
          bbox_to_anchor=(0.5, -0.12),
          ncol=4, frameon=True,
          edgecolor='black', fancybox=False,
          fontsize=6, title_fontsize=6)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig('fig3_cpc.png', dpi=300)
plt.savefig('fig3_cpc.eps')
plt.show()
print("Figure 3 saved")

# Download all figures
from google.colab import files
files.download('fig1_scores.png')
files.download('fig2_scatter.png')
files.download('fig3_cpc.png')
files.download('fig1_scores.eps')
files.download('fig2_scatter.eps')
files.download('fig3_cpc.eps')
print("All figures downloaded.")